In [22]:
# Checking .yaml load/

In [49]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [50]:
from config_handler import initiate_config, load_config

In [51]:
initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8]},
 'method': 'welch',
 'stepSize': 2.7,
 'windowLength': 4}

In [52]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 2.7, 'windowLength': 4}


# TODO: make it so that everything uses the config file:
redo the 'return path' functions, and rely on config file if not given input :)and rely of n config file if not given input :).
prompted gpt 'april 8' so can check that chat at the bottom I think its pretty good 

In [53]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 2.7, 'windowLength': 4}


In [54]:
# TODO: make functions less verbose for data creation
# TODO: clean import statments
# TODO: see if we are calculating Total Energy correctly, I think the whole row is all 0 after std so ... 
# TODO: Experiment with the derivatives and  not derivatives data (preprocessed and 'raw' respectively, I think currently raw_

In [55]:
 # this is a good little tutorial to understand basics of pyspark  
# https://domino.ai/blog/principal-component-analysis-pca-on-large-neuroimaging-datasets-using-pyspark

In [56]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [57]:
# Spark is a library that distributes the load of computation/ram very efficiently and evenly :)

In [58]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

Stopping existing Spark context...
Previous Spark context stopped successfully


In [59]:
# # Set environment variables
# os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# # Create new session with explicit local binding
# spark = SparkSession.builder \
#     .appName("EEG_Analysis") \
#     .config("spark.driver.bindAddress", "127.0.0.1") \
#     .config("spark.driver.host", "127.0.0.1") \
#     .master("local[*]") \
#     .getOrCreate()


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

# print("New Spark session created successfully")

In [60]:
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

New Spark session created successfully


25/04/09 11:32:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [61]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [62]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participants

In [63]:
%%time
# we need to give the path of our  data directory to process the EEG data from
from preprocess_sets import get_data_path

# set_data_path("/Users/user/eeg-ds004504") !!! this doesn't work! so we need to do it manually in preprocess_sets.py! or else won't work!
print(get_data_path())

#Example below is how to get a single subject  and extract its features
sub1 = (
    subject_df
    .filter((subject_df.SubjectID == "sub-001"))
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)
sub1.show()

/Users/admin/eeg-ds004504


/Users/admin/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
Config not found in feature_extraction.py                           (0 + 1) / 1]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 2.7, 'windowLength': 4}
Processing subject sub-001
processSub sub-001
subPath sub-001
subPath data_path /Users/admin/eeg-ds004504
Path handed: /Users/admin/eeg-ds004504/ds004504/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Got 221 epochs for sub-001
<class 'mne.epochs.Epochs'>
Epoch 0
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>
Epoch 1
<class 'mne.epochs.Epochs'>
<class 'mne.epochs.Epochs'>


+---------+-------+--------+---------+--------------------+
|SubjectID|EpochID|WaveBand|Electrode|               Power|
+---------+-------+--------+---------+--------------------+
|  sub-001|   ep-0|   Alpha|      Fp1|0.001279539936847...|
|  sub-001|   ep-0|    Beta|      Fp1| 3.39329353215196E-4|
|  sub-001|   ep-0|   Delta|      Fp1|0.056144485597799276|
|  sub-001|   ep-0|   Theta|      Fp1|0.010567053075609508|
|  sub-001|   ep-0|   Total|      Fp1| 0.00847457627118644|
|  sub-001|   ep-0|   Alpha|      Fp2|0.001107820918021...|
|  sub-001|   ep-0|    Beta|      Fp2|2.940401638078812...|
|  sub-001|   ep-0|   Delta|      Fp2| 0.05968978835575097|
|  sub-001|   ep-0|   Theta|      Fp2|0.007840433533560582|
|  sub-001|   ep-0|   Total|      Fp2|0.008474576271186439|
|  sub-001|   ep-0|   Alpha|       F3|0.001168182585021...|
|  sub-001|   ep-0|    Beta|       F3|3.508409522239829E-4|
|  sub-001|   ep-0|   Delta|       F3| 0.06216100007215241|
|  sub-001|   ep-0|   Theta|       F3|0.

Total rows collected: 20995
Returning DataFrame with 20995 rows
Column names from schema: ['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']
2.339020013809204
                                                                                

In [38]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


# Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = group_a_spark_df.persist()
result_group_c = group_c_spark_df.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

Config not found in feature_extraction.py                         (0 + 12) / 12]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 2.7, 'windowLength': 3}
Processing subject sub-004
processSub sub-004
Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 2.7, 'windowLength': 3}
Processing subject sub-014
processSub sub-014
Config not found in feature_extraction.py
Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'step

Processed 1032175 records for Alzheimer's group
Processed 858040 records for Control group
CPU times: user 171 ms, sys: 103 ms, total: 273 ms
Wall time: 1min 28s


Total rows collected: 32015
Returning DataFrame with 32015 rows
Column names from schema: ['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']
2.844748020172119
                                                                                

In [39]:
result_group_a.columns

['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']

In [40]:
#Since we doni't want to recreate the data all the time, lets save it and I will see you in Example_Data_Processing

In [41]:
type(group_a_spark_df)

pyspark.sql.dataframe.DataFrame

In [42]:
group_a_pandas_df = group_a_spark_df.toPandas() # see here, spark has its own data frame type with lots of its own functions
group_c_pandas_df = group_c_spark_df.toPandas() # Each .pkl is around 50mb last time I checked


In [43]:
group_a_pandas_df.to_pickle("features_alz_extra_features.pkl") # pkl is a way to store python dataframes, its nice
group_c_pandas_df.to_pickle("features_cntrl_extra_features.pkl") # pkl is a way to store python dataframes, its nice

In [44]:
# This is how we would load the .pkl's back in 
# Step 1: Load back into pandas
group_a_pandas_df_loaded = pd.read_pickle("features_alz_extra_features.pkl")
group_c_pandas_df_loaded = pd.read_pickle("features_cntrl_extra_features.pkl")

# Step 2: Convert to Spark DataFrames
group_a_spark_df_loaded = spark.createDataFrame(group_a_pandas_df_loaded)
group_c_spark_df_loaded = spark.createDataFrame(group_c_pandas_df_loaded)

In [45]:
type(group_a_spark_df)

pyspark.sql.dataframe.DataFrame

In [46]:
if group_a_spark_df_loaded.exceptAll(group_a_spark_df).isEmpty(): 
    print("Correctly pkl'd and correcfly loaded into pyspark object")

25/04/09 11:24:14 WARN TaskSetManager: Stage 24 contains a task of very large size (3664 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Correctly pkl'd and correcfly loaded into pyspark object


In [47]:
group_c_spark_df_loaded.select("SubjectID").distinct().orderBy("SubjectID").show(truncate=False)

25/04/09 11:24:17 WARN TaskSetManager: Stage 34 contains a task of very large size (3009 KiB). The maximum recommended task size is 1000 KiB.


+---------+
|SubjectID|
+---------+
|sub-037  |
|sub-038  |
|sub-039  |
|sub-040  |
|sub-041  |
|sub-042  |
|sub-043  |
|sub-044  |
|sub-045  |
|sub-046  |
|sub-047  |
|sub-048  |
|sub-049  |
|sub-050  |
|sub-051  |
|sub-052  |
|sub-053  |
|sub-054  |
|sub-055  |
|sub-056  |
+---------+
only showing top 20 rows



# checking if the config version same as previous version

In [ ]:
import pandas as pd
post_conf_a = pd.read_pickle("features_alz_example_post_config.pkl")
pre_conf_a = pd.read_pickle("features_alz_example.pkl")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd


In [ ]:
post_conf_a = spark.createDataFrame(post_conf_a)
pre_conf_a = spark.createDataFrame(pre_conf_a)

In [ ]:
if post_conf_a.exceptAll(pre_conf_a).isEmpty():
    print("pre and post are equal")